# 🐘 Notebook: Apache Kafka — Architecture & First Steps

[Apache Kafka](https://kafka.apache.org/) is the backend piece that shows up whenever a system needs to move a *lot* of events between services in real time — a purchase, a sensor reading, a page view, a fraud signal — without every producer having to know who's listening. Instead of one service calling another directly, both sides talk to Kafka: producers write events into it, and any number of consumers read them back, independently and at their own pace. That decoupling is what lets large systems scale, evolve, and stay resilient — a slow or crashed consumer never blocks a producer, and adding a new consumer never requires touching existing code.

This first notebook builds the mental model from the ground up: the handful of core terms every Kafka conversation revolves around, how a modern Kafka cluster is actually built (**KRaft**, the architecture that runs *every* Kafka broker since version 4.0), running a real broker plus a web UI in Docker, and writing your first producer and consumer in Python.

> **Event streaming.** An architecture where state changes ("events") are written to a durable, ordered log as they happen, and any number of independent consumers can read that log — live or from the beginning — instead of services calling each other directly.

## 📚 Sources

- [Apache Kafka Documentation](https://kafka.apache.org/documentation/)
- [Kafka: Introduction](https://kafka.apache.org/documentation/#gettingStarted)
- [Kafka: Design](https://kafka.apache.org/documentation/#design)
- [KRaft Documentation](https://kafka.apache.org/documentation/#kraft)
- [Apache Kafka on Docker Hub](https://hub.docker.com/r/apache/kafka)
- [confluent-kafka-python Documentation](https://docs.confluent.io/kafka-clients/python/current/overview.html)
- [confluent-kafka-python on GitHub](https://github.com/confluentinc/confluent-kafka-python)
- [Kafka UI (Provectus) on GitHub](https://github.com/provectus/kafka-ui)

Tip: run each code cell with `Shift + Enter`, in order from top to bottom — later cells reuse the running broker (and topics!) set up earlier.


## 1. Why Kafka exists

Before diving into terminology, it helps to see the problem Kafka solves. Imagine an online shop: when an order is placed, half a dozen things need to happen — charge the customer, update inventory, notify the warehouse, update analytics dashboards, maybe trigger a fraud check. The naive approach has the order service call all five other services directly. That works until:

- One of those services is temporarily down — does the whole order fail?
- You want to add a sixth consumer (say, a recommendation engine) — now you have to modify the order service again.
- Two of those services need to process events at wildly different speeds (fraud detection might take seconds, analytics can batch overnight).

Kafka's answer: the order service publishes an `order-placed` **event** once, to Kafka. Every other service reads that same event independently, at its own pace, without the order service ever knowing (or caring) who's listening. This pattern is called **publish/subscribe** (pub/sub), and it's the foundation of **event-driven architectures**.

> **Event.** A fact about something that happened, at a point in time — usually a small, immutable, timestamped record. Events are written once and (typically) never modified afterwards, which is what makes them safe for many independent readers to share.

Kafka is not the only pub/sub system — RabbitMQ and cloud queues like AWS SQS solve a related problem — but Kafka is specifically built for very **high throughput**, **durable, replayable storage** of events (not just short-lived queues), and **horizontal scalability**. We'll come back to how Kafka compares to those alternatives in the third notebook of this chapter.


## 2. Core concepts

Five terms carry almost the entire vocabulary of Kafka. Get comfortable with these before anything else:

- **Topic** — a named stream of events, e.g. `orders-placed`. Conceptually similar to a table in a database, or a named "channel" — producers write to a topic, consumers read from it.
- **Partition** — a topic is split into one or more partitions, each an independent, strictly **ordered**, append-only log. Splitting a topic into partitions is what allows Kafka to parallelize both writing and reading across multiple machines.
- **Offset** — the position of a record within a partition, a simple increasing integer (0, 1, 2, ...). A consumer tracks "how far it's read" in a partition purely via this number. Crucially, an offset is only ever meaningful *within its own partition* — it is not a global position across a topic, so partition 0's offset 5 and partition 1's offset 5 are two completely unrelated records.
- **Broker** — a single Kafka server process. A **cluster** is a group of brokers working together; each partition of a topic is physically stored on (and served by) one or more brokers.
- **Producer** / **Consumer** — client applications that write records to a topic (producer) or read records from a topic (consumer). Neither talks to the other directly; both only talk to the cluster.

> **Partition = the unit of parallelism and ordering.** Kafka guarantees strict ordering *within* a single partition, but makes **no** ordering guarantee *across* partitions of the same topic. Almost every subtle Kafka bug traces back to forgetting this one sentence.

One more term worth knowing immediately, because it explains *why* a single partition can be read by multiple consumers without conflict: a **consumer group**. Consumers that share the same `group.id` divide up a topic's partitions between them — each partition is only ever read by one consumer *within* the same group at a time, but many different groups can independently and completely re-read the same topic. We'll use this heavily in the next notebook.


## 3. KRaft: how a Kafka cluster manages itself

A Kafka cluster needs one thing beyond just storing your data: a way to agree on **metadata** — which brokers exist, which broker is currently in charge ("leader") of which partition, which topics exist and how they're configured. For years, Kafka delegated that job to a completely separate system, [Apache ZooKeeper](https://zookeeper.apache.org/), meaning every Kafka deployment actually ran *two* distributed systems side by side.

**KRaft** ("Kafka Raft", pronounced "craft") removes that dependency. Since **Kafka 4.0**, ZooKeeper support has been removed entirely — KRaft is not an alternative mode, it is the *only* way a Kafka cluster runs today. Instead of a separate system, Kafka now manages its own metadata internally:

- Cluster metadata (topics, partitions, broker membership, ...) is stored as events in a special internal topic, using the exact same log mechanism Kafka already uses for your data.
- A small subset of brokers act as **controllers** and use the [Raft consensus algorithm](https://raft.github.io/) to agree on the current state of that metadata log, electing one controller as the active leader.
- Every broker keeps a local, continuously-updated copy of the metadata by simply reading that internal topic — the same way a consumer reads any other topic.

> **Why this matters in practice.** One process type to run and monitor instead of two, faster failover (metadata changes propagate as fast as any other Kafka write, rather than through a separate coordination protocol), and a much higher practical limit on how many partitions a cluster can manage. For *us*, running Kafka locally, it means one thing very concretely: a single Docker container is enough to get a fully functional Kafka broker running — no ZooKeeper container to babysit alongside it.

For a single-node development setup like the one we're about to start, one broker plays *both* roles (`broker` and `controller`) at once — in a real production cluster you'd typically dedicate a handful of nodes purely to the controller role instead.


## 4. Running Kafka as a backend service (Docker)

We'll run the official Apache Kafka image directly — a single container is enough thanks to KRaft, no ZooKeeper needed. Pinning an exact version (rather than `:latest`) keeps this notebook reproducible.

Alongside the broker, we'll start a second container: **[Kafka UI](https://github.com/provectus/kafka-ui)**, a free, open-source web dashboard for a Kafka cluster. It's not part of Kafka itself, but it's invaluable while learning — it lets you *see* topics, partitions, consumer groups, and individual messages in a browser instead of only through code or the command line. We'll open it at [http://localhost:18080](http://localhost:18080) once it's running and point it at our broker — deliberately an uncommon port rather than the more familiar `8080`, since that one is already in use by *something* on many development machines (other local web servers, proxies, ...).

> Make sure Docker Desktop is installed and running first — see [setup.md](../../setup.md) if you haven't done this yet.

A couple of notes on the broker configuration below, since Kafka's environment variables look intimidating at first:
- `KAFKA_PROCESS_ROLES=broker,controller` — this single node handles both jobs, appropriate for local development.
- **Two different listeners for two different audiences.** Kafka doesn't just accept a connection on a port — after the initial connection, it tells the client "here's the address to use for every *future* request", taken from `KAFKA_ADVERTISED_LISTENERS`. That address needs to actually be reachable from wherever the client is running, and our two clients are in different places: your Python code runs directly on your machine (needs `localhost:9092`), while the Kafka UI container is a *separate* container that would resolve `localhost` to itself, not to the broker. The fix is to give the broker two listeners — `PLAINTEXT` (advertised as `localhost:9092`, for your machine) and `INTERNAL` (advertised as `kafka-broker:29092`, a hostname only resolvable *inside* Docker) — and put both containers on the same custom Docker network (created below) so that hostname actually resolves.
- `KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093` — tells the controller quorum (just this one node) who its members are, in `nodeId@host:port` form.

If port 9092 or 18080 is already taken on your machine, change the left-hand side of the corresponding `-p` flag below (e.g. `-p 19092:9092`) and update the `bootstrap.servers`/`KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS` values used later in this notebook to match.


In [16]:
# Remove any leftover containers/network from a previous run of this
# notebook, so re-running this cell is always safe.
!docker rm -f kafka-broker kafka-ui > /dev/null 2>&1
!docker network rm kafka-net > /dev/null 2>&1

# A dedicated Docker network so kafka-broker and kafka-ui can resolve each
# other by container name, independent of the host machine's networking.
!docker network create kafka-net > /dev/null

!docker pull -q apache/kafka:4.3.1
!docker run -d --name kafka-broker --network kafka-net -p 9092:9092 \
  -e KAFKA_NODE_ID=1 \
  -e KAFKA_PROCESS_ROLES=broker,controller \
  -e KAFKA_LISTENERS=PLAINTEXT://0.0.0.0:9092,INTERNAL://0.0.0.0:29092,CONTROLLER://0.0.0.0:9093 \
  -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://localhost:9092,INTERNAL://kafka-broker:29092 \
  -e KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093 \
  -e KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,INTERNAL:PLAINTEXT \
  -e KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER \
  -e KAFKA_INTER_BROKER_LISTENER_NAME=PLAINTEXT \
  -e KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1 \
  apache/kafka:4.3.1

import time
time.sleep(5)  # give the broker a moment to finish starting up
!docker ps --filter name=kafka-broker

%3|1789394135.867|FAIL|rdkafka#producer-5| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1789394135.868|FAIL|rdkafka#producer-6| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv6#[::1]:9092 failed: Connection refused (after 0ms in state CONNECT)


docker.io/apache/kafka:4.3.1


%3|1789394136.867|FAIL|rdkafka#producer-5| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv6#[::1]:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1789394136.873|FAIL|rdkafka#producer-6| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)



What's next:
    View a summary of image vulnerabilities and recommendations → docker scout quickview apache/kafka:4.3.1
8a886fd7eaaa40eabfedbc071f2a75b13b1befa85acc323c66335b5aa0b78161


%6|1789394137.873|FAIL|rdkafka#producer-5| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1789394137.873|FAIL|rdkafka#producer-6| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1789394137.930|FAIL|rdkafka#producer-6| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1789394138.034|FAIL|rdkafka#producer-6| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Disconnected: connection reset by peer (after 1ms in state APIVERSION_QUERY, 1 identical error(s) suppressed)


CONTAINER ID   IMAGE                COMMAND                  CREATED         STATUS         PORTS                                         NAMES
8a886fd7eaaa   apache/kafka:4.3.1   "/__cacert_entrypoin…"   5 seconds ago   Up 5 seconds   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   kafka-broker


With the broker up, let's start Kafka UI and point it at it — using the `INTERNAL` listener's address, `kafka-broker:29092`, since Kafka UI is a *separate container* on the `kafka-net` network we just created, not a process on your machine.


In [17]:
!docker pull -q provectuslabs/kafka-ui:v0.7.2
!docker run -d --name kafka-ui --network kafka-net -p 18080:8080 \
  -e KAFKA_CLUSTERS_0_NAME=local \
  -e KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS=kafka-broker:29092 \
  provectuslabs/kafka-ui:v0.7.2

time.sleep(3)
!docker ps --filter name=kafka

docker.io/provectuslabs/kafka-ui:v0.7.2

What's next:
    View a summary of image vulnerabilities and recommendations → docker scout quickview provectuslabs/kafka-ui:v0.7.2
f22175a862a9800ef48d42de14135f9422071516d86ba6445a7fdb0a8cd895fb
CONTAINER ID   IMAGE                           COMMAND                  CREATED          STATUS          PORTS                                           NAMES
f22175a862a9   provectuslabs/kafka-ui:v0.7.2   "/bin/sh -c 'java --…"   3 seconds ago    Up 3 seconds    0.0.0.0:18080->8080/tcp, [::]:18080->8080/tcp   kafka-ui
8a886fd7eaaa   apache/kafka:4.3.1              "/__cacert_entrypoin…"   11 seconds ago   Up 11 seconds   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp     kafka-broker


Open [http://localhost:18080](http://localhost:18080) in your browser now — you should see a cluster named `local` with 0 topics so far. Keep that tab open; we'll come back to it after creating our first topic and producing some messages, so you can watch things appear live instead of only reading printed output here.


## 5. Creating a topic

Topics can be created two ways: implicitly (most Kafka setups **auto-create** a topic the first time something is produced to it) or explicitly, via an **admin client**, which is the safer choice whenever you care about the number of partitions or the replication factor — auto-creation uses whatever cluster-wide defaults are configured, which you usually don't want to rely on.

We'll create a topic explicitly with 3 partitions. Replication factor is set to 1 because our cluster only has a single broker; in a production cluster you'd typically use 3, so each partition survives the loss of up to 2 brokers (see [Notebook 3](./02_3_kafka_ecosystem_and_scaling.ipynb) for more on replication).


In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({"bootstrap.servers": "localhost:9092"})

topic_name = "iot-temperature"
new_topic = NewTopic(topic_name, num_partitions=3, replication_factor=1) # replication_factor=1 is fine for a single-node cluster, but in production you'd want at least 3 nodes and a replication factor of 3.

# create_topics returns a dict of {topic_name: Future} - block on it so we
# know the topic actually exists before moving on.
futures = admin.create_topics([new_topic])
for topic, future in futures.items():
    try:
        future.result()
        print(f"topic '{topic}' created")
    except Exception as e:
        # Re-running this cell after the first time hits "already exists" -
        # that's fine, not an error worth failing on.
        print(f"topic '{topic}' not created: {e}")

print("topics on the cluster:", list(admin.list_topics(timeout=5).topics.keys()))

topic 'iot-temperature' created
topics on the cluster: ['iot-temperature']


## 6. Your first producer and consumer

`confluent-kafka` is the Python client built on top of `librdkafka`, the same high-performance C library that underpins Kafka clients in many other languages — it's the client you'll see in the overwhelming majority of production Python codebases and in interview questions about Kafka client internals.

A **producer** is created once and reused for many `.produce()` calls. Producing is **asynchronous**: `.produce()` just queues the message and returns immediately; a background thread inside the client actually sends it. `.flush()` blocks until every queued message has been either delivered or has failed, which is why we call it before considering the job "done".


In [20]:
from confluent_kafka import Producer
import json

producer = Producer({"bootstrap.servers": "localhost:9092"})

def delivery_report(err, msg):
    """Called once per message, asynchronously, once Kafka has acknowledged
    (or definitively failed) it. This is where you'd log/handle failures in
    a real application - producing is fire-and-forget otherwise."""
    if err is not None:
        print(f"delivery failed: {err}")
    else:
        print(f"delivered to {msg.topic()} [partition {msg.partition()}] @ offset {msg.offset()}")

event = {"sensor_id": 3, "temperature_c": 24.2}
producer.produce(
    topic=topic_name,
    key=str(event["sensor_id"]),          # keys and why they matter: see below
    value=json.dumps(event),
    callback=delivery_report,
)

# flush() blocks until all queued messages are actually sent and acknowledged,
# and triggers any pending delivery callbacks along the way.
producer.flush()

delivered to iot-temperature [partition 1] @ offset 1


0

On the consumer side, `.subscribe()` hands Kafka a list of topics and lets the cluster assign partitions automatically (this is the consumer-group mechanism from section 2 in action, even with just one consumer). `.poll(timeout)` is the main loop primitive: it either returns the next available message or `None` if nothing arrived within `timeout` seconds — real consumers run this in a `while True` loop.

`auto.offset.reset` only matters the *first* time a consumer group reads a topic (before it has ever committed an offset): `"earliest"` starts from the very beginning of the log, `"latest"` (the default) starts from whatever is produced *after* the consumer connects. We use `"earliest"` here so we don't miss the message we just produced.


In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "temperature-readers", # a given message is only delivered to one consumer in a given group
    "auto.offset.reset": "earliest",
})
consumer.subscribe([topic_name])

# Poll a handful of times - in a real app this loop runs forever.
for _ in range(5):
    msg = consumer.poll(timeout=2.0)
    if msg is None:
        continue
    if msg.error():
        print("consumer error:", msg.error())
        continue
    data = json.loads(msg.value())
    print(f"partition {msg.partition()} @ offset {msg.offset()}, key={msg.key().decode()}: {data}")
    break  # we only produced one message so far

consumer.close()  # leaves the consumer group cleanly, triggering an immediate rebalance

partition 1 @ offset 0, key=3: {'sensor_id': 3, 'temperature_c': 24.2}


Head back to the Kafka UI tab: under **Topics → iot-temperature** you should now see 3 partitions and 1 message total, and under **Consumers** the `temperature-readers` group (shown as empty/stable again, since we called `.close()`). This is exactly the workflow you'll use throughout the rest of this chapter — produce or consume from Python, then flip to the UI to see the effect.


## 7. Partitioning and ordering in practice

Section 2 stated the rule: ordering is only guaranteed *within* a partition. Let's make that concrete. Which partition a message lands in is decided by its **key** (unless you specify a partition explicitly): Kafka hashes the key and maps it deterministically to one of the topic's partitions. Two messages with the *same key* always go to the *same partition*, and are therefore always readable in the order they were produced. Messages with *no key* (`key=None`) are spread round-robin across partitions — good for even load distribution, but with **no** ordering guarantee between them at all.

This is why choosing a good partition key is one of the most consequential design decisions in a Kafka-based system: pick something that groups events which *must* stay ordered relative to each other (e.g. all events for the same `sensor_id`, the same `user_id`, the same `order_id`), and let unrelated events land wherever.


In [ ]:
# Produce readings for three different sensors, several times each. Because
# we key by sensor_id, all readings for the same sensor always land on the
# same partition, in order - but there's no guarantee about ordering between
# *different* sensors.
import random

for i in range(9):
    sensor_id = random.choice([1, 2, 3])
    reading = {"sensor_id": sensor_id, "temperature_c": round(20 + random.random() * 10, 1), "seq": i}
    producer.produce(topic=topic_name, key=str(sensor_id), value=json.dumps(reading))
producer.flush()

# Read everything back and print which partition each sensor's readings
# landed on - notice each sensor_id always maps to the same partition.
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "partition-inspector",
    "auto.offset.reset": "earliest", # start at the beginning of the topic so we can see all messages
})
consumer.subscribe([topic_name])

seen = 0
while seen < 10:  # 1 from section 6 + 9 from this cell
    msg = consumer.poll(timeout=2.0)
    if msg is None or msg.error():
        continue
    data = json.loads(msg.value())
    print(f"partition {msg.partition()}  sensor_id={data['sensor_id']}")
    seen += 1

consumer.close()

partition 1  sensor_id=3
partition 1  sensor_id=3
partition 1  sensor_id=2
partition 1  sensor_id=3
partition 1  sensor_id=3
partition 1  sensor_id=2
partition 1  sensor_id=2
partition 1  sensor_id=3
partition 1  sensor_id=3
partition 1  sensor_id=3


## 8. Exercises

### Exercise 1: A topic with more partitions

**Task:** Create a new topic called `clickstream` with 5 partitions and a replication factor of 1 using the `AdminClient`. Confirm it was created by listing all topics on the cluster.


In [ ]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({"bootstrap.servers": "localhost:9092"})
new_topic = NewTopic("clickstream", num_partitions=5, replication_factor=1)

futures = admin.create_topics([new_topic])
for topic, future in futures.items():
    try:
        future.result()
        print(f"topic '{topic}' created")
    except Exception as e:
        print(f"topic '{topic}' not created: {e}")

print(list(admin.list_topics(timeout=5).topics.keys()))
```

</details>


### Exercise 2: Keyed vs. unkeyed messages

**Task:** Produce 10 messages to `clickstream` with `key=None` (i.e. omit the `key` argument entirely), then consume all 10 back and print which partition each one landed on. Do you see an even spread across all 5 partitions, or are some partitions favored? Then repeat, but this time key every message with the *same* fixed key (e.g. `"user-42"`), and confirm they all land on the same single partition.


In [8]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
from confluent_kafka import Consumer

# Part 1: unkeyed messages
for i in range(10):
    producer.produce(topic="clickstream", value=json.dumps({"click": i}))
producer.flush()

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "clickstream-inspector-1",
    "auto.offset.reset": "earliest",
})
consumer.subscribe(["clickstream"])
seen = 0
while seen < 10:
    msg = consumer.poll(timeout=2.0)
    if msg is None or msg.error():
        continue
    print("unkeyed -> partition", msg.partition())
    seen += 1
consumer.close()

# Part 2: same key every time
for i in range(10):
    producer.produce(topic="clickstream", key="user-42", value=json.dumps({"click": i}))
producer.flush()

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "clickstream-inspector-2",
    "auto.offset.reset": "earliest",
})
consumer.subscribe(["clickstream"])
seen = 0
partitions_seen = set()
while seen < 10:
    msg = consumer.poll(timeout=2.0)
    if msg is None or msg.error():
        continue
    if json.loads(msg.value())["click"] < 10:  # ignore leftovers from part 1 if any overlap
        partitions_seen.add(msg.partition())
        seen += 1
consumer.close()
print("partitions used by key 'user-42':", partitions_seen)  # should be exactly one
```

</details>


### Exercise 3: Inspect a topic in Kafka UI

**Task:** No code for this one. In the Kafka UI ([http://localhost:18080](http://localhost:18080)), open the `clickstream` topic, look at its messages, and find where you can see which partition each message is stored in and its offset within that partition. Then look at the **Consumers** page and find the `clickstream-inspector-2` consumer group from Exercise 2 — what state is it shown in now that `.close()` has been called?


In [9]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

In the topic's **Messages** tab, each row shows a `Partition` and `Offset` column directly. In **Consumers**, a group that has called `.close()` on every member (and isn't actively polling) shows as `EMPTY` — it still exists (Kafka remembers its committed offsets), but has no active members currently assigned to any partition. It would return to `STABLE` the moment a new consumer with that `group.id` subscribes again.

</details>


## Cleanup

If you're moving straight on to [Notebook 2](./02_2_kafka_features.ipynb), you can leave the containers running — it reuses the same broker. Otherwise, tear everything down:


In [10]:
!docker rm -f kafka-broker kafka-ui
!docker network rm kafka-net

kafka-broker


kafka-ui


%6|1789393835.240|FAIL|rdkafka#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 10206ms in state UP)
%6|1789393835.240|FAIL|rdkafka#producer-2| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 10157ms in state UP)


kafka-net
